# IOAI — 2026 Contest Star Galaxy Quasar (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-star-galaxy-quasar/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Cosmic Probe — Star / Galaxy / Quasar 분류 (APOAI 2026)

SDSS 5-밴드 광도(u,g,r,i,z)로 천체를 **STAR / GALAXY / QSO** 3분류한다. 불균형 문제(QSO ≈ 18.8%)이고,
평가지표는 **F2 score**(recall 강조 — 희귀 QSO 를 놓치는 것이 더 큰 손해). `data/train.csv`(라벨 포함)로
학습하고 `data/testdata.csv`(라벨 없음)를 예측해 `submission.csv`(`objid,type`)를 만든다.
채점: 클래스별 F2 의 참개수-가중 평균.

In [ ]:
import pandas as pd
train = pd.read_csv("data/train.csv")            # objid, ra, dec, modelMag_*, type
test  = pd.read_csv("data/testdata.csv")         # (type 없음)
MAGS = ["modelMag_u","modelMag_g","modelMag_r","modelMag_i","modelMag_z"]
print(train.shape, "| classes:", train.type.value_counts().to_dict())

In [ ]:
# 베이스라인: 다수 클래스(GALAXY) 예측 — F2 낮음
pred = ["GALAXY"] * len(test)
pd.DataFrame({"objid": test["objid"], "type": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)